<a href="https://colab.research.google.com/github/raz0208/Natural-Language-Processing-Practices/blob/main/TopicModelling/EmbeddingsAnalysis_TopicClustering_all_mpnet_base_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topic Clustring

## **Clustering Topic Models from TurfTopic**
 - Top2Vec
 - BERTopic

In [ ]:
# Install libraries and packages
!pip install 'turftopic[umap-learn, datamapplot]'

In [2]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch

In [4]:
# Read and Load dataset
dataset = pd.read_csv('sample_PubMedDataAbstracts.csv')

# Show the datasets
### Abstract Embeddings Sample Dataset
print('Node Content:', dataset.shape)
print(dataset)

Node Content: (10000, 4)
      Unnamed: 0                                              title  \
0              0  Phenotypic variability of Niemann-Pick disease...   
1              1  Recurrent hypoglycemia secondary to metformin ...   
2              2  Adaptation of the Ambulatory and Home Care Rec...   
3              3  Multidimensional family therapy in adolescents...   
4              4  Balanced crystalloids versus isotonic saline i...   
...          ...                                                ...   
9995        9995  Methylmercury in Industrial Harbor Sediments i...   
9996        9996  Factors Affecting Secondhand Smoke Avoidance B...   
9997        9997  Predicting Infectious Disease Using Deep Learn...   
9998        9998  Diosgenin Glucoside Protects against Spinal Co...   
9999        9999  Omics Approaches for Engineering Wheat Product...   

                                               abstract  year  
0     Background Niemann-Pick disease type C (NPC) i...  2

In [5]:
# Extract only the 'abstract' column and drop others
abstracts = dataset['abstract'].dropna().reset_index(drop=True)

# Display a few samples to verify
print(abstracts)

0       Background Niemann-Pick disease type C (NPC) i...
1       Background Metformin toxicity is well known to...
2       Background Measuring service use and costs is ...
3       Background Substance use and delinquency are c...
4       Objectives Intravenous fluids are one of the m...
                              ...                        
9995    The distribution of methylmercury (MeHg) and t...
9996    The purpose of this study was to examine the s...
9997    Infectious disease occurs when a person is inf...
9998    Spinal cord injury (SCI) is a severe traumatic...
9999    Abiotic stresses greatly influenced wheat prod...
Name: abstract, Length: 10000, dtype: object


### 1. TurfTopic Default model and configuration
  - Use "all-MiniLM-L6-v2" to extract embeddings
  - Use Top2Vec to topic modelling ans clustering

In [6]:
# Import topic clustring required libraries
from sentence_transformers import SentenceTransformer
from turftopic import Top2Vec

In [ ]:
# Using TurfTopic default encoder to extract embedding of the dataset
encoder = SentenceTransformer("all-mpnet-base-v2")
embeddings = encoder.encode(abstracts, show_progress_bar=True)

In [ ]:
# Show embeddings matrix and Check the dimention of each eambeding
print(embeddings,"\n\n", embeddings.shape)

In [ ]:
# Training model (Uses HDBSCAN and umap)
model = Top2Vec(encoder=encoder, random_state=42)
topic_data = model.prepare_topic_data(abstracts, embeddings=embeddings)

In [ ]:
model.print_topics()

In [ ]:
# Cluster model hierarchy
model.hierarchy.cut(3).plot_tree()

In [ ]:
# Merging topics to reduce number of topics
model.reduce_topics(n_reduce_to=25)
print(model.hierarchy.cut(3))

In [ ]:
# Model hierarchy after merging topics
fig = model.hierarchy[190].plot_tree()
fig.show()

In [ ]:
# We will reset the hierarchy, so that we can see all topics at once.
model.reset_topics()
fig = model.plot_clusters_datamapplot(hover_text=dataset["title"])
fig.show()

## **2. TurfTopic Default model and configuration**
  - Use "all-MiniLM-L6-v2" to extract embeddings
  - Use BERTopic to topic modelling ans clustering

In [ ]:
# Show embeddings matrix and Check the dimention of each eambeding
print(embeddings,"\n\n", embeddings.shape)

In [ ]:
# import BERTopic library
from turftopic import BERTopic

In [ ]:
# Training model (Uses HDBSCAN and umap)
model1 = BERTopic(encoder=encoder, random_state=42)
topic_data = model1.prepare_topic_data(abstracts, embeddings=embeddings)

In [ ]:
model1.print_topics()

In [ ]:
# Cluster model hierarchy
model1.hierarchy.cut(3).plot_tree()

In [ ]:
# Merging topics to reduce number of topics
model1.reduce_topics(n_reduce_to=25)
print(model1.hierarchy.cut(3))

In [ ]:
# Model hierarchy after merging topics
fig = model1.hierarchy[340].plot_tree()
fig.show()

In [ ]:
# We will reset the hierarchy, so that we can see all topics at once.
model1.reset_topics()
fig = model1.plot_clusters_datamapplot(hover_text=dataset["title"])
fig.show()

### **Visualisation**
- Dendrogram

In [ ]:
# Required libraries for dendrogram
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
# Get the reduced topic-term matrix and topic labels
topic_matrix = topic_data.topic_term_matrix  # shape: (n_topics, vocab_size)
topic_labels = topic_data.topic_names        # list of strings or keywords per topic

# # Optional: turn list of keywords into single string labels
# labels = [' | '.join(words[:3]) for words in topic_labels]

# Compute hierarchical clustering
linkage_matrix = linkage(topic_matrix, method='ward')

# Plot dendrogram
plt.figure(figsize=(12, 8))
dendrogram(linkage_matrix, labels=None, leaf_rotation=90, leaf_font_size=10)
plt.title("Topic Hierarchy Dendrogram")
plt.tight_layout()
plt.show()

In [ ]:
# Extract topic names and embeddings from the model
topic_names = topic_data.topic_names
topic_embeddings = topic_data.topic_term_matrix  # shape: (num_topics, vocab_size)

topic_names, topic_embeddings

In [ ]:
# Optional: use document_topic_matrix if you'd rather visualize topics by their document composition
# topic_embeddings = model.document_topic_matrix.T  # shape: (num_topics, num_documents)

# Compute pairwise cosine distance matrix
distance_matrix = cosine_distances(topic_embeddings)

# Step 3: Perform hierarchical clustering
linkage_matrix = linkage(distance_matrix, method='ward')  # 'ward' works better with Euclidean, use 'average' for cosine

# Step 4: Plot the dendrogram
plt.figure(figsize=(20, 8))
dendrogram(linkage_matrix, labels=[f"Topic {i}" for i in range(len(topic_names))], leaf_rotation=90, leaf_font_size=10)
plt.title("Dendrogram of Vec2Top Topics")
plt.xlabel("Topics")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()